# Scale100K confirmation: R6 / R7

Train only `R6_100k_all3_ce25` and `R7_100k_harmful_gated_worst` using notebook 04's frozen manifests and 100K recipe. Existing C/P3 are optional result references.

1. Attach the completed 00d pooled resource bundle, CelebA/LCC images, MiniFASNet source/checkpoint and notebook 04's frozen `manifests/` directory.
2. Edit paths in the config. No resampling, detector or contract-source setup is required.
3. Run both with `RUN_FILTER=None`, or use `['R6_100k_all3_ce25']` / `['R7_100k_harmful_gated_worst']` in separate Kaggle sessions.
4. Val15K selects the checkpoint and calibrates the threshold. Full CelebA Test and LCC train/dev/eval/pooled run only after freeze.
5. Optional reference paths are read after evaluation. They never affect training; missing references remain pending.

GPU training/inference are performed on Kaggle, not during notebook creation. Resume is per run from epoch-level `last_state.pth` in `OUTPUT_ROOT`.


In [ ]:
# Edit Kaggle paths here. Mount existing frozen 100K/15K manifests separately.
RESOURCE_ROOT='/kaggle/input/datasets/maithanhphuong/full-resources-v3/minifasnet_binary_crossdomain_resources_v2_lcc_pooled'
CELEBA_ROOT='/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof'
LCC_ROOT='/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD'
MINIFASNET_SOURCE_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py'
MINIFASNET_CHECKPOINT_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth'
SCALE_MANIFEST_ROOT='/kaggle/input/datasets/maithanhphuong/100k-output/binary_crossdomain_scale100k_C_P3_Q3_v1/manifests'
EXISTING_SCALE100K_ROOT=None  # Optional results only; never used for training.
EXISTING_FULL_LCC_ROOT=None  # Optional notebook 05 evaluated C/P3 outputs.
MICRO_R6_R7_SUMMARY=None  # Optional completed 10K comparison_summary.csv.
OUTPUT_ROOT='/kaggle/working/binary_crossdomain_scale100k_R6_R7_v1'
RUN_FILTER= ['R6_100k_all3_ce25']  # None = all variants in this notebook; or a list of exact run IDs.
RESUME=True
NUM_WORKERS=4
AMP=True
DATA_SEED=43
TRAIN_SEED=100
BATCH_SIZE=256
MAX_EPOCHS=20
MILESTONES=[6,14]
GAMMA=.2
EARLIEST_EARLY_STOP_EPOCH=8
EARLY_STOP_PATIENCE=4
WARMUP_EPOCHS=1
CSMR_BAND_CENTERS=[.15,.45,.75]
CSMR_BAND_SIGMA=.10
CSMR_PRESERVE_DC=True
SELECTION_CHUNK_SIZE=32
CROP_SCALE=2.7
INPUT_SIZE=80
METRIC_TIE_EPS=1e-12

RUNS={
 "R6_100k_all3_ce25":{"method":"ALL3","clean_ce_weight":.75,"spectral_ce_weight":.25},
 "R7_100k_harmful_gated_worst":{"method":"HARMFUL","clean_ce_weight":.75,"spectral_ce_weight":.25},
}


In [ ]:
PROTOCOL_TEXT="# Scale100K R6/R7 confirmation, protocol v1\n\n## Question and scope\n\nConfirm the selected all-band CE and harmful-gated worst mechanisms at the existing C/P3 100K scale. Train exactly `R6_100k_all3_ce25` and `R7_100k_harmful_gated_worst`. Existing `C100k_copy_pretrained` and `P3_100k_worst_ce25` are read-only references; no C/P3/Q3 or additional mechanism is trained. LCC remains an external-development domain. CASIA is reserved for later confirmation and is not accessed here.\n\n## Frozen controls\n\nRead `scale100k_train.csv` and `scale15k_val.csv` directly from `SCALE_MANIFEST_ROOT`, which must point to notebook 04's existing frozen manifest directory. Do not resample, rebuild, or derive splits. Data seed is 43 and train seed is 100. Use the completed 00d resource bundle for CelebA localization/auxiliary labels and LCC localization/manifests. Copy the two frozen manifest files into the output for portability without changing their content.\n\nReuse notebook 04's MiniFASNetV2 wrapper, 128-D representation, binary PAD head and 11/5/40 auxiliary heads. Load its compatible official PAD-pretrained feature weights, skipping the historical classifier and initializing project heads with seed 100. Record the initial state SHA and require the two runs to start from the same state. Preserve notebook 04's data-order generator and worker/epoch augmentation seed policy, with an independent bbox RNG. Spectral gains use a separately seeded generator for each run and cannot consume the base augmentation RNG.\n\nKeep frozen raw SCRFD boxes, jitter probability 0.20 with scale 0.95–1.05 and shift ±0.05 face size, 2.7× crop, 80×80 BGR float input divided by 255, and the existing HorizontalFlip/ISONoise/brightness-contrast/MotionBlur recipe. No normalization, mixup, cutmix, label smoothing or weighted sampler is added. Evaluation has no jitter or augmentation, and no detector runs.\n\nUse SGD with lr 0.005, momentum 0.9 and weight decay 5e-4; batch 256; at most 20 epochs; MultiStepLR milestones [6,14], gamma 0.2, stepped after each epoch. LR is 0.005 for epochs 1–6, 0.001 for 7–14 and 0.0002 for 15–20. Val15K minimum ACER selects the checkpoint, breaking ties with higher AUC. Earliest early stopping is epoch 8, with four active epochs without improvement. Both runs warm up for one clean epoch, excluded from best selection and patience.\n\n## Mechanisms and loss\n\nUse LOW/MID/HIGH centers [0.15,0.45,0.75], sigma 0.10 and one gain per sample from Uniform(0.65,0.90), shared across bands/channels. Preserve DC and phase; do not amplify or tune spectral parameters.\n\n```\nL_aux = 0.1 L_spoof_type + 0.1 L_lighting + L_attributes\nL_R6 = 0.75 CE_clean + 0.25 mean(CE_LOW, CE_MID, CE_HIGH) + L_aux\nL_R7 = 0.75 CE_clean + 0.25 CE_harmful + L_aux\n```\n\nAuxiliary supervision is clean-view only and attributes are Real-only. Warm-up loss is `CE_clean + L_aux`.\n\nR6 supervises all three views unconditionally and uses the tested separate train-mode band forwards to preserve its BatchNorm updates. R7 temporarily uses `eval()` and `no_grad()` to evaluate clean and three bands. Its label-aligned margin is `z_real-z_attack` for Real and its negative for Attack. Select the smallest band margin and detach the gate `m_worst < m_clean`. Restore the previous training mode, forward the selected view differentiably and average its per-sample CE over harmful samples. Zero harmful samples gives a differentiable zero. No SmoothL1, margin regression, KL or random-band loss is used.\n\n## Evaluation, diagnostics and resume\n\nEvaluate Val15K only inside the epoch loop. Reload best, recompute the Val15K min-ACER threshold and write `evaluation_freeze.json` before CelebA full Test or LCC inference. Full Test is required in this scale-confirmation round. Use one model-specific locked threshold for Test and LCC training/development/evaluation/combined. Retain failed localization candidates as unscored; report candidate/scored counts and coverage. Raw LCC counts are 8299/2948/7580/18827. Concatenate the three physical LCC prediction tables and recompute pooled metrics; do not average metrics.\n\nReport AUC, EER, TPR@FPR1%, APCER, BPCER, ACER/HTER, accuracy and threshold for Val15K, Test and each LCC population. R6 diagnostics include clean/all3/per-band CE, clean/per-band aligned margins, gain and total loss. R7 diagnostics include harmful CE/count/fraction, worst-band rates, clean/worst margin, margin drop, flips, gain and total loss.\n\nEach run resumes independently from an epoch-level state containing model, optimizer, scheduler, scaler, epoch, best checkpoint payload, patience, histories and RNG states. Run/config/initial-state checks prevent cross-run resume. A stored best checkpoint payload recovers the committed best state if interruption occurred while writing a newer epoch.\n\n## Artifacts and comparison\n\nUse `/kaggle/working/binary_crossdomain_scale100k_R6_R7_v1/`, with independent `runs/<run_id>/` checkpoints, config, initialization audit, histories, diagnostics, frozen threshold, metrics and `predictions/`. Save this protocol, `comparison_summary.csv/json` and `scale100k_R6_R7_report.md` at root. `RUN_FILTER` can select either run for separate Kaggle sessions; each run produces its artifacts and run-specific ZIP without waiting for the other.\n\n`EXISTING_SCALE100K_ROOT` is optional and is read only after new-run evaluation. Its C/P3 Val/Test/official-LCC results can enter a four-way table without loading or training their models. Optional `EXISTING_FULL_LCC_ROOT` supplies notebook 05's separate pooled-evaluation outputs; unavailable split metrics remain pending rather than inferred. Optional `MICRO_R6_R7_SUMMARY` supplies measured 10K results for descriptive cross-scale deltas. Reference paths never gate training. No automatic final winner or extra search is selected.\n\nValidate parsing/compilation, run IDs, reused source pipeline, initialization equality, R6/R7 loss and detached BN-safe gating with small checks before expensive Kaggle execution. A single training seed and missing optional reference populations limit comparisons; no unmeasured scale-retention claim is made.\n"

import hashlib, shutil, importlib.util, inspect, json, math, os, random, zipfile
from collections import Counter, OrderedDict
from pathlib import Path
import albumentations as A
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader
from IPython.display import FileLink, display

_BBOX_RNG=random.Random(TRAIN_SEED+500000)
AUG_SEED=TRAIN_SEED
BBOX_SEED=TRAIN_SEED+500000
GAIN_SEED=TRAIN_SEED+11000
GAIN_RANGE=(.65,.90)
SPLITS=('training','development','evaluation')
EXPECTED_LCC={'training':8299,'development':2948,'evaluation':7580,'combined':18827}
if RUN_FILTER is not None and (not isinstance(RUN_FILTER,(list,tuple)) or not RUN_FILTER or len(set(RUN_FILTER))!=len(RUN_FILTER) or set(RUN_FILTER)-set(RUNS)):
    raise ValueError('RUN_FILTER contains an unknown or duplicate run ID')
if NUM_WORKERS<1:raise ValueError('NUM_WORKERS must be >= 1')
if (BATCH_SIZE,MAX_EPOCHS,MILESTONES,GAMMA,TRAIN_SEED)!=(256,20,[6,14],.2,100):raise RuntimeError('Frozen scale100K recipe changed')
root=Path(RESOURCE_ROOT);out=Path(OUTPUT_ROOT);source=Path(MINIFASNET_SOURCE_PATH);pretrained=Path(MINIFASNET_CHECKPOINT_PATH)
for path in (source,pretrained,Path(SCALE_MANIFEST_ROOT)/'scale100k_train.csv',Path(SCALE_MANIFEST_ROOT)/'scale15k_val.csv',
             root/'manifests/crossdomain_celeba_source_dev_full.csv',root/'bbox/celeba_bbox_cache_full.json',
             root/'labels/celeba_auxiliary_labels.csv'):
    if not path.is_file():raise FileNotFoundError(path)
def read_json(path):return json.loads(Path(path).read_text())
def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,sort_keys=True,allow_nan=False)+'\n')
def save_torch(path,value):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    torch.save(value,tmp);os.replace(tmp,path)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()
def state_sha(state):
    h=hashlib.sha256()
    for k in sorted(state):
        v=state[k].detach().cpu().contiguous();h.update(k.encode());h.update(str(v.dtype).encode());h.update(str(tuple(v.shape)).encode());h.update(v.numpy().tobytes())
    return h.hexdigest()
def require_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD' or rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):raise RuntimeError('Unscorable/non-SCRFD: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Not a raw bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite(box).all() or x2<=x1 or y2<=y1:raise RuntimeError('Invalid bbox: '+key)
    return rec

def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return cv2.resize(patch,(size,size))

# The literature's augmentation ranges are fixed. Support both documented Albumentations APIs.
# ISONoise works in RGB/HSV; convert only within augmentation, then restore BGR for MiniFASNet.
iso_params=inspect.signature(A.ISONoise).parameters
brightness_params=inspect.signature(A.RandomBrightnessContrast).parameters
blur_params=inspect.signature(A.MotionBlur).parameters
if {'color_shift','intensity'}<=set(iso_params) and {'brightness_limit','contrast_limit'}<=set(brightness_params) and 'blur_limit' in blur_params:
    AUGMENTATION_API='legacy_limits'
    iso=A.ISONoise(color_shift=(.15,.35),intensity=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_limit=.2,contrast_limit=.2,brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_limit=5,p=.2)
elif {'color_shift_range','intensity_range'}<=set(iso_params) and {'brightness_range','contrast_range'}<=set(brightness_params) and 'blur_range' in blur_params:
    AUGMENTATION_API='range_parameters'
    iso=A.ISONoise(color_shift_range=(.15,.35),intensity_range=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_range=(-.2,.2),contrast_range=(-.2,.2),brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_range=(3,5),p=.2)
else:raise RuntimeError('Unsupported Albumentations augmentation API; freeze and audit a compatible version')
COPY_AUG=A.Compose([A.HorizontalFlip(p=.5),iso,bright,motion])
def copied_aug_bgr(patch):
    rgb=cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)
    augmented=COPY_AUG(image=rgb)['image']
    return cv2.cvtColor(augmented,cv2.COLOR_RGB2BGR)
def tensor_of(patch):
    if patch.shape!=(80,80,3) or patch.dtype!=np.uint8:raise ValueError('Expected BGR uint8 80x80')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def pad_score(logits):
    z=torch.as_tensor(logits)
    if z.shape[1]==3:return (z[:,0]-torch.logsumexp(z[:,1:],dim=1)).float().cpu().numpy()
    if z.shape[1]==2:return (z[:,0]-z[:,1]).float().cpu().numpy()
    raise ValueError('Expected two or three PAD logits')
def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:raise ValueError('Val needs all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0);nr=int(real.sum());na=len(y)-nr
    bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)];rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]
def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}

def radial_masks(size=INPUT_SIZE,device=None):
    if device is None:device=globals()['device']
    # Same radial geometry as the tested 03b CSMR notebook.
    y=torch.arange(size,device=device,dtype=torch.float32)-size//2
    yy,xx=torch.meshgrid(y,y,indexing='ij')
    radius=torch.sqrt(xx.square()+yy.square())/math.sqrt(2*(size//2)**2)
    centers=torch.as_tensor(CSMR_BAND_CENTERS,dtype=torch.float32,device=device)
    masks=torch.exp(-((radius[None]-centers[:,None,None])**2)/(2*CSMR_BAND_SIGMA**2))
    if masks.shape!=(3,size,size) or not torch.isfinite(masks).all():raise RuntimeError('Invalid CSMR radial masks')
    return masks
def spectral_views(x,gains,masks):
    if x.ndim!=4 or x.shape[1:]!=(3,80,80) or not torch.isfinite(x).all():raise ValueError('CSMR input')
    x=x.float();gains=gains.float().reshape(-1)
    if len(gains)!=len(x) or not ((gains>=0)&(gains<=1)).all():raise ValueError('CSMR gain')
    spectrum=torch.fft.fftshift(torch.fft.fft2(x,norm='ortho'),dim=(-2,-1))
    fields=1-(1-gains[:,None,None,None])*masks[None]
    if CSMR_PRESERVE_DC:fields[:,:,40,40]=1
    raw=torch.fft.ifft2(torch.fft.ifftshift(spectrum[:,None]*fields[:,:,None],dim=(-2,-1)),norm='ortho').real
    clips=((raw<0)|(raw>1)).float().mean(dim=(2,3,4));views=raw.clamp(0,1)
    if not torch.isfinite(views).all():raise RuntimeError('Nonfinite CSMR views')
    return views,clips
def aligned_margin(logits,binary):
    d=logits[:,0].float()-logits[:,1].float()
    return torch.where(binary==0,d,-d)

def auxiliary_loss(outputs,binary,spoof,lighting,attrs):
    pad,spoof_logits,light_logits,attr_logits=outputs
    ls=F.cross_entropy(spoof_logits,spoof);ll=F.cross_entropy(light_logits,lighting);mask=binary==0
    la=F.binary_cross_entropy_with_logits(attr_logits[mask],attrs[mask]) if mask.any() else attr_logits.sum()*0
    return ls,ll,la
def restore_rng(state,gain_rng):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda']);gain_rng.set_state(state['gain']);_BBOX_RNG.setstate(state['bbox'])
def bbox_jitter(rec):
    x1,y1,x2,y2=map(float,rec['bbox_xyxy']);w,h=x2-x1,y2-y1
    if _BBOX_RNG.random()<.20:
        base=max(w,h);cx,cy=(x1+x2)/2,(y1+y2)/2;scale=_BBOX_RNG.uniform(.95,1.05)
        cx+=_BBOX_RNG.uniform(-.05,.05)*base;cy+=_BBOX_RNG.uniform(-.05,.05)*base;w*=scale;h*=scale
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]
    return [x1,y1,x2,y2]


def load_frame(name):return pd.read_csv(root/'manifests'/(name+'.csv'),keep_default_na=False)
train=pd.read_csv(Path(SCALE_MANIFEST_ROOT)/'scale100k_train.csv',keep_default_na=False)
val=pd.read_csv(Path(SCALE_MANIFEST_ROOT)/'scale15k_val.csv',keep_default_na=False)
test=load_frame('crossdomain_celeba_source_dev_full')
if (len(train),len(val),len(test))!=(100000,15000,67170):raise RuntimeError('Unexpected frozen CelebA populations')
if train.sample_key.duplicated().any() or val.sample_key.duplicated().any() or set(train.sample_key)&set(val.sample_key):
    raise RuntimeError('Train/Val sample overlap')
if set(train.subject_id.astype(str))&set(val.subject_id.astype(str)):raise RuntimeError('Train/Val subject overlap')
for frame in (train,val):
    if set(frame.sampling_seed)!={DATA_SEED}:raise RuntimeError('Frozen data seed differs')
    if not frame.usable_for_pad.astype(bool).all() or set(frame.bbox_origin)!={'SCRFD'} or set(frame.three_class_label)!={0,1,2}:
        raise RuntimeError('Frozen scale split is not fully usable')
records=read_json(root/'bbox/celeba_bbox_cache_full.json')['records']
aux=pd.read_csv(root/'labels/celeba_auxiliary_labels.csv',keep_default_na=False).set_index('sample_key',verify_integrity=True)
for frame in (train,val,test):
    if not set(frame.sample_key)<=set(records) or not set(frame.sample_key)<=set(aux.index):raise RuntimeError('Missing CelebA cache/aux entry')
lcc_frames={};lcc_records={}
for split in ('training','development','evaluation','combined'):
    name='lcc_official_evaluation_full' if split=='evaluation' else 'lcc_'+split+'_full'
    cache_name='lcc_official_evaluation_bbox_cache' if split=='evaluation' else 'lcc_'+split+'_bbox_cache'
    path=root/'manifests'/(name+'.csv');cache=root/'bbox'/(cache_name+'.json')
    if not path.is_file() or not cache.is_file():raise FileNotFoundError((path,cache))
    frame=pd.read_csv(path,keep_default_na=False)
    if len(frame)!=EXPECTED_LCC[split] or frame.path.duplicated().any():raise RuntimeError('Unexpected LCC population: '+split)
    recs=read_json(cache)['records'] if split!='combined' else None
    if recs is not None and set(frame.path)!=set(recs):raise RuntimeError('LCC manifest/cache keys differ: '+split)
    lcc_frames[split]=frame;lcc_records[split]=recs
ordered_pool=pd.concat([lcc_frames[s].assign(lcc_split=s) for s in SPLITS],ignore_index=True)
if not lcc_frames['combined'].equals(ordered_pool):raise RuntimeError('Combined LCC manifest differs from physical splits')
if not torch.cuda.is_available():raise RuntimeError('Kaggle CUDA GPU required for the frozen scale100K batch size')
source_spec=importlib.util.spec_from_file_location('local_minifasnet_v2',source)
module=importlib.util.module_from_spec(source_spec);source_spec.loader.exec_module(module)
device=torch.device('cuda');use_amp=bool(AMP)
out.mkdir(parents=True,exist_ok=True);(out/'runs').mkdir(exist_ok=True)
(out/'manifests').mkdir(exist_ok=True)
for filename in ('scale100k_train.csv','scale15k_val.csv'):
    shutil.copyfile(Path(SCALE_MANIFEST_ROOT)/filename,out/'manifests'/filename)
(out/'SCALE100K_R6_R7_PROTOCOL_v1.md').write_text(PROTOCOL_TEXT)
MANIFEST_HASHES={f:sha(Path(SCALE_MANIFEST_ROOT)/f) for f in ('scale100k_train.csv','scale15k_val.csv')}
print('GPU:',torch.cuda.get_device_name(0),'Train/Val:',len(train),len(val),'LCC:',{s:len(f) for s,f in lcc_frames.items()})

def init_model(spec):
    seed_all(TRAIN_SEED);model=PADNet(spec['copied']);audit={'initialization':spec['init'],'checkpoint_sha256':sha(pretrained),'source_sha256':sha(source)}
    if spec['init']=='pretrained':
        raw=torch.load(pretrained,map_location='cpu',weights_only=True)
        if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):raise RuntimeError('Official checkpoint format mismatch')
        stripped=OrderedDict((k[7:],v) for k,v in raw.items())
        if tuple(stripped['prob.weight'].shape)!=(3,128) or tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):raise RuntimeError('Official checkpoint architecture mismatch')
        target=model.trunk.state_dict();compatible={k:v for k,v in stripped.items() if k!='prob.weight'}
        bad=[k for k,v in compatible.items() if k not in target or tuple(v.shape)!=tuple(target[k].shape)]
        absent=[k for k in target if k not in compatible and k!='prob.weight']
        if bad or absent:raise RuntimeError(f'Pretrained key mismatch: {bad[:5]} {absent[:5]}')
        missing,unexpected=model.trunk.load_state_dict(compatible,strict=False)
        if unexpected or (spec['copied'] and missing) or (not spec['copied'] and set(missing)!={'prob.weight'}):raise RuntimeError('Pretrained load mismatch')
        # Reinitialize historical classifier too; copied heads were initialized by seed 100.
        if not spec['copied']:model.trunk.prob.reset_parameters()
        audit.update({'loaded_keys':sorted(compatible),'skipped_keys':['prob.weight'],'missing_keys':list(missing),'unexpected_keys':list(unexpected)})
    else:audit.update({'loaded_keys':[],'skipped_keys':[],'missing_keys':[],'unexpected_keys':[]})
    audit['initial_state_sha256']=state_sha(model.state_dict())
    with torch.no_grad():
        model.eval();assert model(torch.zeros(2,3,80,80)).shape==(2,2 if spec['copied'] else 3)
    return model,audit

initial_model,initial_audit=init_model({'init':'pretrained','copied':True})
INITIAL_STATE_SHA=initial_audit['initial_state_sha256']
del initial_model

class CelebaData(Dataset):
    def __init__(self,frame,training=False,copied=False):self.rows=frame.reset_index(drop=True);self.training=training;self.copied=copied
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.sample_key);rec=require_record(records[key],key)
        image=cv2.imread(str(Path(CELEBA_ROOT)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(key)
        patch=mini_crop_bgr(image,bbox_jitter(rec) if self.training else rec['bbox_xyxy'])
        if self.training:patch=copied_aug_bgr(patch) if self.copied else mild_aug(patch)
        a=aux.loc[key]
        return tensor_of(patch),int(row.three_class_label),int(row.binary_label),int(a.spoof_type),int(a.lighting),torch.tensor([float(a[f'attr_{j:02d}']) for j in range(40)]),key

class LCCData(Dataset):
    def __init__(self,split):
        self.split=split;self.rows=lcc_frames[split].loc[lcc_frames[split].usable_for_pad.astype(bool)].reset_index(drop=True)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.path);rec=require_record(lcc_records[self.split][key],key)
        path=(Path(LCC_ROOT)/'LCC_FASD_evaluation'/key) if self.split=='evaluation' else Path(LCC_ROOT)/key
        image=cv2.imread(str(path),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(path)
        return tensor_of(mini_crop_bgr(image,rec['bbox_xyxy'])),int(row.label),key

def loader(ds,batch=BATCH_SIZE,shuffle=False,epoch=0):
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    def worker_init(wid):
        global _BBOX_RNG
        s=TRAIN_SEED+epoch*1000+wid;random.seed(s);np.random.seed(s);torch.manual_seed(s)
        _BBOX_RNG=random.Random(s+500000)
        if hasattr(COPY_AUG,'set_random_seed'):COPY_AUG.set_random_seed(s)
    return DataLoader(ds,batch_size=batch,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=True,drop_last=False,generator=gen,worker_init_fn=worker_init)

def infer(model,dl,lcc_mode=False):
    model.eval();keys=[];ys=[];zs=[]
    with torch.no_grad():
        for batch in dl:
            if lcc_mode:x,y,k=batch
            else:x,y,_,_,_,_,k=batch
            with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))
            keys.extend(k);ys.extend(y.numpy().tolist());zs.append(z.float().cpu().numpy())
    if not zs:raise RuntimeError('No scorable evaluation samples')
    return list(map(str,keys)),np.asarray(ys,dtype=int),np.concatenate(zs)

def select_worst_view(model,base,binary,masks,gain_rng):
    was_training=model.training;model.eval()
    chosen=[];bands=[];gains_out=[];clips_out=[];clean_margins=[];worst_margins=[]
    try:
        with torch.no_grad():
            all_gains=torch.empty(len(base),device=base.device).uniform_(*GAIN_RANGE,generator=gain_rng)
            for start in range(0,len(base),SELECTION_CHUNK_SIZE):
                x=base[start:start+SELECTION_CHUNK_SIZE];y=binary[start:start+SELECTION_CHUNK_SIZE]
                gains=all_gains[start:start+len(x)]
                with torch.cuda.amp.autocast(enabled=use_amp):clean_z=model(x)
                clean_margins.append(aligned_margin(clean_z,y).detach())
                views,clips=spectral_views(x,gains,masks);margins=[]
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=use_amp):z=model(views[:,band])
                    margins.append(aligned_margin(z,y))
                margin=torch.stack(margins,dim=1);ix=margin.argmin(dim=1)
                row=torch.arange(len(x),device=x.device)
                chosen.append(views[row,ix].detach());bands.append(ix.detach())
                gains_out.append(gains.detach());clips_out.append(clips[row,ix].detach())
                worst_margins.append(margin[row,ix].detach())
    finally:model.train(was_training)
    return tuple(torch.cat(seq) for seq in (chosen,bands,gains_out,clips_out,clean_margins,worst_margins))


def training_terms(model,x,binary,spoof,lighting,attrs,spec,masks,gain_rng,active):
    method=spec['method'];n=len(x);selected=None;bands=None;gains=None;select_clean=None;select_worst=None
    if active and method=='HARMFUL':
        selected,bands,gains,_,select_clean,select_worst=select_worst_view(model,x,binary,masks,gain_rng)
    elif active and method=='ALL3':
        gains=torch.empty(n,device=x.device).uniform_(*GAIN_RANGE,generator=gain_rng)
        with torch.no_grad():all_views,_=spectral_views(x,gains,masks)
    with torch.cuda.amp.autocast(enabled=use_amp):
        clean=model(x,aux=True);clean_ce=F.cross_entropy(clean[0],binary)
        ls,ll,la=auxiliary_loss(clean,binary,spoof,lighting,attrs);aux_total=.1*ls+.1*ll+la
        spectral_ce=clean_ce*0;harmful_count=0;spectral_logits=None;all3_logits=None;band_ce=[]
        if active and method=='HARMFUL':
            spectral_logits=model(selected)
            ce_each=F.cross_entropy(spectral_logits,binary,reduction='none')
            harmful=(select_worst<select_clean).detach()
            harmful_count=int(harmful.sum())
            spectral_ce=(ce_each*harmful).sum()/harmful_count if harmful_count else spectral_logits.sum()*0
        elif active and method=='ALL3':
            # Separate train forwards retain the tested micro BatchNorm semantics.
            all3_logits=[model(all_views[:,band]) for band in range(3)]
            band_ce=[F.cross_entropy(z,binary) for z in all3_logits]
            spectral_ce=sum(band_ce)/3
        loss=.75*clean_ce+.25*spectral_ce+aux_total if active else clean_ce+aux_total
    with torch.no_grad():
        cm=aligned_margin(clean[0],binary).mean()
        flips=int(((clean[0].argmax(1)==binary)&(spectral_logits.argmax(1)!=binary)).sum()) if spectral_logits is not None else 0
        scalars={'train_total_loss':float(loss.detach()),'clean_ce':float(clean_ce.detach()),
                 'spectral_ce':float(spectral_ce.detach()),'aux_loss':float(aux_total.detach()),
                 'clean_aligned_margin':float(select_clean.mean()) if select_clean is not None else float(cm),
                 'worst_margin':float(select_worst.mean()) if select_worst is not None else 0.,
                 'margin_drop':float((select_clean-select_worst).mean()) if select_worst is not None else 0.,
                 'harmful_count':harmful_count,'harmful_fraction':harmful_count/n,
                 'harmful_ce':float(spectral_ce.detach()) if method=='HARMFUL' else 0.,
                 'all3_ce':float(spectral_ce.detach()) if method=='ALL3' else 0.,
                 'spectral_flip_rate':flips/n,'sampled_gain_mean':float(gains.mean()) if gains is not None else 0.}
        for i,label in enumerate(('low','mid','high')):
            scalars[label+'_ce']=float(band_ce[i]) if band_ce else 0.
            scalars[label+'_aligned_margin']=float(aligned_margin(all3_logits[i],binary).mean()) if all3_logits is not None else 0.
    counts=Counter(bands.detach().cpu().tolist()) if bands is not None else Counter()
    return loss,scalars,counts

def snapshot_rng(gain_rng):
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all(),'gain':gain_rng.get_state(),'bbox':_BBOX_RNG.getstate()}
def restore_rng(state,gain_rng,random_gain_rng,band_rng):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda']);gain_rng.set_state(state['gain'])
    random_gain_rng.set_state(state['random_gain']);band_rng.set_state(state['band'])

def run_config(name,spec):
    return {'run_id':name,'spec':spec,'resource_root':str(root),'train_manifest':'scale100k_train.csv',
      'val_manifest':'scale15k_val.csv','manifest_sha256':MANIFEST_HASHES,'data_seed':DATA_SEED,
      'train_seed':TRAIN_SEED,'initial_state_sha256':INITIAL_STATE_SHA,
      'base_order':'torch Generator(TRAIN_SEED+epoch)','worker_seed':'TRAIN_SEED+epoch*1000+worker_id',
      'bbox_rng_seed':'worker_seed+500000','gain_rng_seed':GAIN_SEED,
      'pretrained_checkpoint':str(pretrained),'model_source':str(source),
      'celeba_root':CELEBA_ROOT,'lcc_root':LCC_ROOT,'num_workers':NUM_WORKERS,
      'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'milestones':MILESTONES,'gamma':GAMMA,
      'optimizer':{'name':'SGD','lr':.005,'momentum':.9,'weight_decay':5e-4},
      'warmup_epochs':WARMUP_EPOCHS,'earliest_early_stop_epoch':EARLIEST_EARLY_STOP_EPOCH,'patience':EARLY_STOP_PATIENCE,
      'spectral':{'centers':CSMR_BAND_CENTERS,'sigma':CSMR_BAND_SIGMA,'gain':[.65,.90],'preserve_dc':CSMR_PRESERVE_DC},
      'input':'80x80 BGR float [0,1] 2.7x','augmentation_api':AUGMENTATION_API,
      'albumentations_version':A.__version__,'amp':use_amp}

def prediction_frame(frame,keys,y,z,threshold,keycol):
    scores=pad_score(z)
    scored=pd.DataFrame({keycol:keys,'pad_logit_score':scores,'pred_real':scores>=threshold,
                         'logit_0':z[:,0],'logit_1':z[:,1]})
    result=frame.merge(scored,on=keycol,how='left',validate='one_to_one',sort=False)
    valid=frame.usable_for_pad.astype(bool)
    if result.loc[valid,'pad_logit_score'].isna().any() or result.loc[~valid,'pad_logit_score'].notna().any():
        raise RuntimeError('Prediction accounting differs from frozen candidates')
    result['locked_threshold']=threshold
    return result

def evaluate_lcc(model,folder,locked):
    scored_predictions=[]
    for split in SPLITS:
        frame=lcc_frames[split]
        keys,y,z=infer(model,loader(LCCData(split)),True)
        expected=frame.loc[frame.usable_for_pad.astype(bool),'path'].astype(str).tolist()
        if keys!=expected:raise RuntimeError('LCC scored order differs: '+split)
        pred=prediction_frame(frame,keys,y,z,locked,'path').assign(lcc_split=split)
        mm=metrics(y,pad_score(z),locked,len(frame))
        write_json(folder/f'lcc_{split}_metrics.json',mm)
        pred.to_csv(folder/'predictions'/f'lcc_{split}_predictions.csv',index=False)
        scored_predictions.append(pred)
    pooled=pd.concat(scored_predictions,ignore_index=True)
    if not pooled[['path','label','status','usable_for_pad','bbox_origin','lcc_split']].equals(
        lcc_frames['combined'][['path','label','status','usable_for_pad','bbox_origin','lcc_split']]):
        raise RuntimeError('Combined prediction candidates differ from 00d manifest')
    usable=pooled.usable_for_pad.astype(bool)
    mm=metrics(pooled.loc[usable,'label'],pooled.loc[usable,'pad_logit_score'],locked,len(pooled))
    write_json(folder/'lcc_combined_metrics.json',mm)
    pooled.to_csv(folder/'predictions'/'lcc_combined_predictions.csv',index=False)

def evaluate_celeba_test(model,folder,locked):
    scored=test.loc[test.usable_for_pad.astype(bool)]
    keys,y,z=infer(model,loader(CelebaData(scored)))
    if keys!=scored.sample_key.astype(str).tolist():raise RuntimeError('CelebA Test order differs')
    write_json(folder/'test_metrics.json',metrics(y,pad_score(z),locked,len(test)))
    prediction_frame(test,keys,y,z,locked,'sample_key').to_csv(folder/'predictions'/'test_predictions.csv',index=False)

def run_one(name,spec):
    folder=out/'runs'/name;folder.mkdir(parents=True,exist_ok=True);(folder/'predictions').mkdir(exist_ok=True)
    cfg=run_config(name,spec);cfg_path=folder/'config.json'
    if cfg_path.is_file() and read_json(cfg_path)!=cfg:raise RuntimeError(name+' existing config differs')
    write_json(cfg_path,cfg)
    best_path=folder/'best.pth';last_path=folder/'last_state.pth';done_path=folder/'completion.json'
    if done_path.is_file():
        needed=[folder/(k+'_metrics.json') for k in ('val','test','lcc_training','lcc_development','lcc_evaluation','lcc_combined')]
        if not all(p.is_file() for p in needed) or not best_path.is_file():
            raise RuntimeError(name+' incomplete completion outputs')
        print(name,'completed; skipping')
        return
    model,audit=init_model({'init':'pretrained','copied':True})
    if audit['initial_state_sha256']!=INITIAL_STATE_SHA:raise RuntimeError('Initial state differs')
    write_json(folder/'initialization_audit.json',audit)
    model=model.to(device)
    optimizer=torch.optim.SGD(model.parameters(),lr=.005,momentum=.9,weight_decay=5e-4)
    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=MILESTONES,gamma=GAMMA)
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    gain_rng=torch.Generator(device=device).manual_seed(GAIN_SEED)
    best_checkpoint=None
    history=[];diagnostics=[];best_acer=float('inf');best_auc=-float('inf');best_epoch=0;no_improve=0;start=1
    if last_path.is_file():
        if not RESUME:raise RuntimeError(name+' has a saved state; enable RESUME or use a new output root')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if state.get('run_id')!=name or state.get('config')!=cfg:raise RuntimeError(name+' resume run/config differs')
        best_checkpoint=state['best_checkpoint']
        if state['best_epoch']:
            if best_checkpoint['run_id']!=name or best_checkpoint['epoch']!=state['best_epoch'] or best_checkpoint['config']!=cfg:raise RuntimeError('Resume best payload differs')
            save_torch(best_path,best_checkpoint)
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        history=state['history'];diagnostics=state['diagnostics'];best_acer=state['best_acer']
        best_auc=state['best_auc'];best_epoch=state['best_epoch'];no_improve=state['no_improve']
        start=state['epoch']+1
        if len(history)!=state['epoch']:raise RuntimeError(name+' resume history length differs')
        restore_rng(state['rng'],gain_rng)
        print(name,'resume at epoch',start)
    val_dl=loader(CelebaData(val))
    masks=radial_masks()
    if start<=MAX_EPOCHS and not (history and history[-1]['stop_reached']):
        for epoch in range(start,MAX_EPOCHS+1):
            expected_lr=.005 if epoch<=6 else (.001 if epoch<=14 else .0002)
            if not math.isclose(optimizer.param_groups[0]['lr'],expected_lr,rel_tol=1e-9):raise RuntimeError('Scheduler LR differs')
            warmup=spec['method']!='CLEAN' and epoch<=WARMUP_EPOCHS
            active=not warmup
            model.train();totals=Counter();band_count=Counter();seen=0;harmful_count_total=0
            train_ds=CelebaData(train,training=True,copied=True)
            for x,_,binary,spoof,lighting,attrs,_ in loader(train_ds,shuffle=True,epoch=epoch):
                x=x.to(device,non_blocking=True);binary=binary.to(device);spoof=spoof.to(device)
                lighting=lighting.to(device);attrs=attrs.to(device);optimizer.zero_grad(set_to_none=True)
                loss,scalars,bands=training_terms(model,x,binary,spoof,lighting,attrs,
                    spec,masks,gain_rng,active)
                scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
                count=len(x);seen+=count
                for key,value in scalars.items():totals[key]+=value*count
                band_count.update(bands)
                harmful_count_total+=scalars['harmful_count']
            keys,y,z=infer(model,val_dl)
            if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Val order differs')
            scores=pad_score(z);threshold=threshold_from_val(y,scores)
            vm=metrics(y,scores,threshold,len(val))
            improved=active and (vm['acer']<best_acer-METRIC_TIE_EPS or
                (abs(vm['acer']-best_acer)<=METRIC_TIE_EPS and vm['auc']>best_auc+METRIC_TIE_EPS))
            if active:
                if improved:
                    best_acer=vm['acer'];best_auc=vm['auc'];best_epoch=epoch;no_improve=0
                    best_checkpoint={'run_id':name,'epoch':epoch,'config':cfg,
                        'val_acer':best_acer,'val_auc':best_auc,
                        'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}}
                    save_torch(best_path,best_checkpoint)
                else:no_improve+=1
            row={'epoch':epoch,'phase':'warmup' if warmup else 'active','lr':optimizer.param_groups[0]['lr'],
                 'eligible_for_final_checkpoint':active,'val_acer':vm['acer'],'val_auc':vm['auc'],
                 'val_eer':vm['eer'],'val_threshold':threshold,'best_epoch':best_epoch,
                 'no_improve_count':no_improve,'clean_ce_weight':1.0 if warmup else spec['clean_ce_weight'],
                 'spectral_ce_weight':0.0 if warmup else spec['spectral_ce_weight'],
                 'method':'CLEAN_WARMUP' if warmup else spec['method'],
                 **{k:v/seen for k,v in totals.items()},
                 **{f'{label}_count':band_count[i] for i,label in enumerate(('low','mid','high'))},
                 **{f'{label}_fraction':band_count[i]/seen for i,label in enumerate(('low','mid','high'))}}
            row['harmful_count']=harmful_count_total
            if active and spec['method']!='CLEAN':diagnostics.append(dict(row))
            stop=epoch==MAX_EPOCHS or (active and epoch>=EARLIEST_EARLY_STOP_EPOCH and no_improve>=EARLY_STOP_PATIENCE)
            row['stop_reached']=stop;history.append(row)
            scheduler.step()
            state={'run_id':name,'config':cfg,'epoch':epoch,
                'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                'scaler':scaler.state_dict(),'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                'no_improve':no_improve,'rng':snapshot_rng(gain_rng),'best_checkpoint':best_checkpoint,
                'history':history,'diagnostics':diagnostics}
            save_torch(last_path,state)
            pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
            if spec['method']!='CLEAN':pd.DataFrame(diagnostics).to_csv(folder/'spectral_diagnostics.csv',index=False)
            print(name,'epoch',epoch,'Val ACER',vm['acer'],'best',best_epoch,'patience',no_improve,flush=True)
            if stop:break
    if not best_path.is_file() or best_epoch<= (0 if spec['method']=='CLEAN' else WARMUP_EPOCHS):
        raise RuntimeError(name+' has no eligible best checkpoint')
    best=torch.load(best_path,map_location='cpu',weights_only=True)
    if best['run_id']!=name or best['config']!=cfg or best['epoch']!=best_epoch:
        raise RuntimeError(name+' best checkpoint identity differs')
    model.load_state_dict(best['state_dict']);model.eval()
    keys,y,z=infer(model,val_dl)
    if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Frozen Val order differs')
    locked=threshold_from_val(y,pad_score(z))
    vm=metrics(y,pad_score(z),locked,len(val))
    if abs(vm['acer']-best['val_acer'])>1e-6 or abs(vm['auc']-best['val_auc'])>1e-6:
        raise RuntimeError('Best checkpoint does not reproduce Val selection metrics')
    write_json(folder/'val_metrics.json',vm)
    prediction_frame(val,keys,y,z,locked,'sample_key').to_csv(folder/'predictions'/'val_predictions.csv',index=False)
    freeze={'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,'val_metrics':vm}
    freeze_path=folder/'evaluation_freeze.json'
    if freeze_path.is_file() and read_json(freeze_path)!=freeze:raise RuntimeError('Evaluation freeze differs')
    write_json(freeze_path,freeze)
    evaluate_celeba_test(model,folder,locked)
    evaluate_lcc(model,folder,locked)
    write_json(done_path,{'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,
                          'initial_state_sha256':INITIAL_STATE_SHA})
    del model
    torch.cuda.empty_cache()


In [ ]:
SCORE_SPLITS=('val','test','lcc_training','lcc_development','lcc_evaluation','lcc_combined')
TABLE_COLUMNS=('val_acer','val_auc','test_auc','test_acer','lcc_training_auc','lcc_training_hter',
 'lcc_development_auc','lcc_development_hter','lcc_evaluation_auc','lcc_evaluation_eer','lcc_evaluation_hter',
 'lcc_combined_auc','lcc_combined_eer','lcc_combined_hter','lcc_combined_tpr_at_fpr_1pct')
def result_row(name,folder):
    done=read_json(folder/'completion.json')
    row={'run_id':name,'status':'complete','selected_epoch':done['best_epoch']}
    for split in SCORE_SPLITS:
        for key,value in read_json(folder/(split+'_metrics.json')).items():row[split+'_'+key]=value
    return row

def metric_table(rows):
    lines=['| Run | '+' | '.join(TABLE_COLUMNS)+' |', '|---|'+'---:|'*len(TABLE_COLUMNS)]
    for row in rows:
        lines.append('| '+row['run_id']+' | '+' | '.join(
            f'{row[c]:.6f}' if isinstance(row.get(c),(int,float)) else 'pending' for c in TABLE_COLUMNS)+' |')
    return lines

def make_zip(path,files,base):
    with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for file in sorted(files):
            if file.is_file():archive.write(file,(Path(base.name)/file.relative_to(base)).as_posix())

def export_summary():
    rows=[result_row(name,out/'runs'/name) for name in RUNS if (out/'runs'/name/'completion.json').is_file()]
    write_json(out/'comparison_summary.json',rows)
    pd.DataFrame(rows).to_csv(out/'comparison_summary.csv',index=False)
    report=['# Scale100K R6/R7 confirmation','',
      'Frozen Train100K/Val15K, seed 100. Best selected only on Val15K; one warm-up epoch excluded.',
      'CelebA full Test and full LCC use the frozen Val15K threshold. LCC combined metrics are recomputed from concatenated predictions.',
      'No automatic method winner. Missing runs remain pending.','',*metric_table(rows),'',
      'Candidate/scored counts, coverage and all threshold-based metrics are in comparison_summary.csv/json and each run metric JSON.',
      'R6 band CE/margins and R7 harmful counts, margins, band fractions and flips are in spectral_diagnostics.csv.',
      'Single seed results are descriptive; this notebook does not run a search or a new target calibration.']
    (out/'scale100k_R6_R7_report.md').write_text('\n'.join(report)+'\n')
    return rows

selected=[name for name in RUNS if RUN_FILTER is None or name in RUN_FILTER]
for name in selected:
    try:run_one(name,RUNS[name])
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('Frozen batch 256 ran out of GPU memory; stop this run') from exc
    export_summary()  # Independent artifacts even if the next run is interrupted.
    run_folder=out/'runs'/name
    run_zip=out.parent/(name+'.zip')
    make_zip(run_zip,run_folder.rglob('*'),run_folder)
    display(FileLink(str(run_zip)))
rows=export_summary()

# Read old result files only after the new runs have completed. Never load old models.
notes=[]
def load_reference(name,old_root):
    folder=old_root/'runs'/name
    row={'run_id':name,'status':'reference'}
    freeze_path=folder/'evaluation_freeze.json'
    freeze=read_json(freeze_path) if freeze_path.is_file() else {}
    locked=freeze.get('locked_threshold')
    row['selected_epoch']=freeze.get('selected_epoch',freeze.get('best_epoch'))
    for split in SCORE_SPLITS:
        candidates=[folder/(split+'_metrics.json')]
        if split=='lcc_evaluation':candidates.append(folder/'lcc_metrics.json')
        if EXISTING_FULL_LCC_ROOT and split.startswith('lcc_'):
            extra=Path(EXISTING_FULL_LCC_ROOT)
            candidates=[extra/name/(split+'_metrics.json'),extra/'runs'/name/(split+'_metrics.json')]+candidates
        path=next((p for p in candidates if p.is_file()),None)
        if path is None:continue
        mm=read_json(path)
        if locked is not None and mm.get('locked_source_threshold')!=locked:
            notes.append(name+' '+split+': skipped because reference threshold differs from its own Val freeze.')
            continue
        for key,value in mm.items():row[split+'_'+key]=value
    # Existing summaries can provide fields if individual metric files are unavailable.
    summary=old_root/'comparison_summary.json'
    if summary.is_file():
        previous=next((r for r in read_json(summary) if r.get('run_id')==name),{})
        for key,value in previous.items():
            if key.startswith(('val_','test_')) or key=='selected_epoch':row.setdefault(key,value)
            elif key.startswith('lcc_') and not key.startswith(('lcc_training_','lcc_development_','lcc_evaluation_','lcc_combined_')):
                row.setdefault('lcc_evaluation_'+key[4:],value)
    if not any(key in row for key in TABLE_COLUMNS):notes.append(name+': result files unavailable.')
    return row

def differences(left,right,lookup,columns):
    a=lookup.get(left,{});b=lookup.get(right,{})
    result=[f'{c} {a[c]-b[c]:+.6f}' for c in columns if isinstance(a.get(c),(int,float)) and isinstance(b.get(c),(int,float))]
    return '; '.join(result) if result else 'pending the required measured results'

if EXISTING_SCALE100K_ROOT and Path(EXISTING_SCALE100K_ROOT).is_dir():
    old_root=Path(EXISTING_SCALE100K_ROOT)
    refs=[load_reference(name,old_root) for name in ('C100k_copy_pretrained','P3_100k_worst_ce25')]
    new_lookup={r['run_id']:r for r in rows}
    four=refs+[new_lookup.get(name,{'run_id':name,'status':'pending'}) for name in RUNS]
    write_json(out/'final_4way_comparison.json',four)
    pd.DataFrame(four).to_csv(out/'final_4way_comparison.csv',index=False)
    lookup={r['run_id']:r for r in four}
    report=['# C / P3 / R6 / R7 scale100K comparison','',
      'C/P3 are read-only measured references; no reference is retrained. Each model uses its own Val15K threshold.',
      'Missing full-LCC reference splits remain pending. Optional notebook 05 outputs supply these populations.','',
      *metric_table(four),'','## Questions','']
    micro={}
    if MICRO_R6_R7_SUMMARY and Path(MICRO_R6_R7_SUMMARY).is_file():
        micro={r['run_id']:r for r in pd.read_csv(MICRO_R6_R7_SUMMARY).to_dict('records')}
    for question,scale_name,micro_name,columns in [
        ('Q1. R6 balanced advantage after scale-up','R6_100k_all3_ce25','R6_all3_ce25',('val_acer','val_auc','lcc_combined_auc','lcc_combined_hter')),
        ('Q2. R7 cross-domain AUC after scale-up','R7_100k_harmful_gated_worst','R7_harmful_gated_worst',('lcc_training_auc','lcc_development_auc','lcc_evaluation_auc','lcc_combined_auc'))]:
        if micro_name in micro:
            cross={scale_name:lookup.get(scale_name,{}),micro_name:micro[micro_name]}
            report.append(question+': 100K minus 10K: '+differences(scale_name,micro_name,cross,columns)+'. This scale change alone does not establish an advantage over a control.')
        else:report.append(question+': pending measured 10K results (MICRO_R6_R7_SUMMARY). No retention claim is inferred.')
    cols=('val_acer','test_acer','lcc_evaluation_auc','lcc_evaluation_hter','lcc_combined_auc','lcc_combined_hter')
    report+=['Q3. Is adaptive selection necessary? R6 minus P3: '+differences('R6_100k_all3_ce25','P3_100k_worst_ce25',lookup,cols)+'. Interpret source and target trade-offs together.',
             'Q4. Does harmful filtering help? R7 minus P3: '+differences('R7_100k_harmful_gated_worst','P3_100k_worst_ce25',lookup,cols)+'. Inspect harmful diagnostics alongside these deltas.',
             'Q5. Stability across domains: see the splitwise table above; the following ranks are descriptive.','']
    for split in SCORE_SPLITS:
        for metric,ascending in (('auc',False),('hter',True)):
            key=split+'_'+metric
            measured=[r for r in four if isinstance(r.get(key),(int,float))]
            if len(measured)==4:
                ranked=sorted(measured,key=lambda r:r[key],reverse=not ascending)
                report.append('- '+key+': '+' > '.join(r['run_id']+f' ({r[key]:.6f})' for r in ranked)+'. Best-to-worst; equal values are ties.')
            else:report.append('- '+key+': four-way rank pending missing populations.')
    for split in ('test','lcc_training','lcc_development','lcc_evaluation','lcc_combined'):
        counts={(r.get(split+'_candidate_n'),r.get(split+'_scored_n')) for r in four if split+'_candidate_n' in r}
        if len(counts)>1:notes.append(split+': reference candidate/scored populations differ; interpret comparisons cautiously.')
    report+=['',*notes,'','Single-seed descriptive comparison. No automatic final method winner is declared.']
    (out/'final_4way_report.md').write_text('\n'.join(report)+'\n')
else:
    print('Optional C/P3 comparison skipped: EXISTING_SCALE100K_ROOT is None or unavailable.')
zip_path=out.parent/(out.name+'.zip')
make_zip(zip_path,out.rglob('*'),out)
print('Completed',len(rows),'of',len(RUNS),'new runs')
display(FileLink(str(zip_path)))
